In [1]:
import os

import sys
import pandas as pd
import numpy as np

In [2]:
curr_dir = os.getcwd()
parent_dir = os.path.dirname(curr_dir)
proj_dir = os.path.dirname(parent_dir)
sys.path.append(proj_dir)

In [3]:
from utility.data_log_functions import DataLogHelper

In [6]:
def compare_multiple_code_generation_logs(res_dir: str, filter: str = None):

    if filter is None:
        csv_logs = [f for f in os.listdir(res_dir) if (os.path.isfile(os.path.join(res_dir, f)) and f.endswith(".csv"))]
    else:
        csv_logs = [f for f in os.listdir(res_dir) if (os.path.isfile(os.path.join(res_dir, f)) and f.endswith(".csv") and filter in f)]

    count = 0
    res = []

    log_file_names = [csv_file_name.replace('.csv', '') for csv_file_name in csv_logs]

    results_df = pd.DataFrame(columns=log_file_names, index = log_file_names)
    for file_name in log_file_names:
        results_df.loc[file_name, file_name] = float('nan')

    while count < len(csv_logs):
        log1_file_name = csv_logs.pop()
        for log2_file_name in csv_logs:
            log1_file_path = os.path.join(res_dir, log1_file_name)
            log2_file_path = os.path.join(res_dir, log2_file_name)

            log1 = pd.read_csv(log1_file_path)
            log2 = pd.read_csv(log2_file_path) 

            log1_inconsistencies, log2_inconsistencies = DataLogHelper.compare_code_generation_dataframe_results(log1=log1, log2=log2)
            print(f"log1: {log1_file_name} > {log1_inconsistencies}")
            print(f"log2: {log2_file_name} > {log2_inconsistencies}")

            results_df.loc[log1_file_name.replace('.csv', ''), log2_file_name.replace('.csv', '')] = log1_inconsistencies
            results_df.loc[log2_file_name.replace('.csv', ''), log1_file_name.replace('.csv', '')] = log2_inconsistencies

        count += 1
    
    return results_df

In [7]:
res_dir = proj_dir + "/results/code_generation/mistral_old"

res = compare_multiple_code_generation_logs(res_dir=res_dir, filter="random")
res_df = pd.DataFrame(res)

print(res_df)

log1: mistral-small-2506_one_shot_random.csv > 8
log2: mistral-small-2506_zero_shot_random.csv > 26
log1: mistral-small-2506_one_shot_random.csv > 17
log2: mistral-small-2506_few_shot_random.csv > 9
log1: mistral-small-2506_few_shot_random.csv > 10
log2: mistral-small-2506_zero_shot_random.csv > 36
                                    mistral-small-2506_zero_shot_random  \
mistral-small-2506_zero_shot_random                                 NaN   
mistral-small-2506_few_shot_random                                   10   
mistral-small-2506_one_shot_random                                    8   

                                    mistral-small-2506_few_shot_random  \
mistral-small-2506_zero_shot_random                                 36   
mistral-small-2506_few_shot_random                                 NaN   
mistral-small-2506_one_shot_random                                  17   

                                    mistral-small-2506_one_shot_random  
mistral-small-2506_zero_shot_